# Installation
Install the core dependencies

In [35]:
!pip install deepagents langsmith slack-sdk langchain-core langchain_groq langchain-google-genai

In [2]:
from google.colab import userdata

In [3]:
LANGSMITH_TRACING = userdata.get("LANGSMITH_TRACING")
LANGSMITH_API_KEY = userdata.get("LANGSMITH_API_KEY")

# Setup the backend
Deep Agents use backends to execute code in sandboxed environments.

We create a sandbox on LangSmith to the agent execute comands and create and access files on a sandbox; not in our machine.

In [4]:
import os

from deepagents.backends.langsmith import LangSmithSandbox
from langsmith.sandbox import SandboxClient

In [5]:
os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
os.environ["LANGSMITH_TRACING"] = LANGSMITH_TRACING

client = SandboxClient(api_key=LANGSMITH_API_KEY)
sandbox = client.create_sandbox()
backend = LangSmithSandbox(sandbox=sandbox)

# Upload sample data
create an upload sample sales data to the backend

In [6]:
import csv
import io

In [7]:
# Create sample sales data
data = [
    ["Date", "Product", "Units Sold", "Revenue"],
    ["2025-08-01", "Widget A", 10, 250],
    ["2025-08-02", "Widget B", 5, 125],
    ["2025-08-03", "Widget A", 7, 175],
    ["2025-08-04", "Widget C", 3, 90],
    ["2025-08-05", "Widget B", 8, 200],
]

# Convert to CSV bytes with IO
text_buf= io.StringIO()
writer = csv.writer(text_buf)
writer.writerows(data)
csv_bytes = text_buf.getvalue().encode("utf-8")
text_buf.close()

# Upload to backend
backend.upload_files([("/root/data/sales_data.csv", csv_bytes)])

[FileUploadResponse(path='/root/data/sales_data.csv', error=None)]

We can also upload the data on a csv format with the lib "pandas".

```
import pandas as pd

df = pd.DataFrame(data[1:], collumns=data[0])
csv_bytes = df.to_csv(index=False).encode("utf-8")
```

# Implement custom tools


In [36]:
from langchain.tools import tool
from langchain.agents.middleware import TodoListMiddleware
from langchain_core.utils.uuid import uuid7
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import InMemorySaver
from deepagents import create_deep_agent
from langchain_google_genai import ChatGoogleGenerativeAI

from slack_sdk import WebClient
import smtplib
from email.message import EmailMessage
import ssl
import mimetypes
from pathlib import PurePosixPath

In [46]:
sender = userdata.get("GMAIL_EMAIL")
gmail_password = userdata.get("GMAIL_PASSWORD")
reciever = sender

@tool("send-email", description="Send a email with the sales data information")
def gmail_send_message(subject: str, content: str, file_path: str | None = None) -> str:
  email = EmailMessage()
  email["Subject"] = subject
  email["From"] = sender
  email["To"] = reciever
  email.set_content(content)

  if file_path:
    _file = backend.download_files([file_path])[0]

    if _file.error or _file.content is None:
       return f"Falha ao obter o anexo: {_file.error or file_path}"

    name = PurePosixPath(file_path).name
    mime_type = mimetypes.guess_type(name)[0] or "application/octet-stream"
    _type, subtype = mime_type.split("/", 1)

    email.add_attachment(
        _file.content,
        maintype=_type,
        subtype=subtype,
        filename=name
    )


  with smtplib.SMTP_SSL("smtp.gmail.com", 465, context=ssl.create_default_context(), timeout=30) as gmail:
    gmail.login(sender, gmail_password)
    gmail.send_message(email)

  return f"Email enviado com sucesso para {reciever} | Subject do email: {subject}!"

In [49]:
checkpoint = InMemorySaver()

llm = ChatGoogleGenerativeAI(
    api_key=userdata.get("GOOGLE_API_KEY"),
    model="gemini-3.5-flash-lite"
)

agent_1 = create_deep_agent(
    model=llm,
    tools=[gmail_send_message],
    checkpointer=checkpoint,
    backend=backend,
    middleware=[TodoListMiddleware()]
)

thread_id = str(uuid7())
config = {"configurable": {"thread_id": thread_id}}

In [50]:
input_message = {
    "role": "user",
    "content": (
        "Analise ./data/sales_data.csv no diretório atual e gere um gráfico bonito. Quando terminar, envie sua análise e o gráfico para o Gmail usando a ferramenta."
    ),
}

stream = agent_1.stream_events(
    {"messages": input_message},
    config,
    version='v3'
)

for snapshot in stream.values:
    snapshot["messages"][-1].pretty_print()

================================ Human Message =================================

Analise ./data/sales_data.csv no diretório atual e gere um gráfico bonito. Quando terminar, envie sua análise e o gráfico para o Gmail usando a ferramenta.
================================== Ai Message ==================================

[{'type': 'tool_call', 'id': 'call_16032208508922354689', 'name': 'ls', 'args': {'path': '/workspace'}}]
Tool Calls:
  ls (call_16032208508922354689)
 Call ID: call_16032208508922354689
  Args:
    path: /workspace
================================= Tool Message =================================
Name: ls

Error: Path '/workspace': path_not_found
================================== Ai Message ==================================

[{'type': 'tool_call', 'id': 'call_5172469179471198939', 'name': 'ls', 'args': {'path': '/'}}]
Tool Calls:
  ls (call_5172469179471198939)
 Call ID: call_5172469179471198939
  Args:
    path: /
================================= Tool Message ==========

GoogleAPIError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}